# Stage 3 -- Model-Ready: Combined Means

## Input
- `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_daily_means.parquet` -- already z-scored daily means table
- `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_monthly_means.parquet` -- already z-scored monthly means table

## Purpose
Merges the z-scored daily and monthly means tables into a single combined dataset by forward-filling monthly features to daily frequency using `merge_asof`. Both inputs are already z-standardised -- no additional standardisation is applied here. On each trading day the model sees: today's z-scored daily features (stock cwmean + macro daily), plus the most recent month-end's z-scored monthly features (stock cwmean + macro monthly), plus both a daily and a monthly target.

---

## Pipeline

### Step 1: Load Both Z-Scored Tables
Both the daily and monthly means tables are loaded and sorted by date. Shape and date ranges are reported for both.

### Step 2: Prefix Monthly Columns
All monthly feature columns (everything except `date` and `target_monthly_return`) are prefixed with `monthly_`. This serves two purposes: avoiding column name conflicts with daily features, and making the frequency origin of each feature unambiguous in the combined table. Conflicts between daily and monthly column names are checked and reported before prefixing. The prefix is applied regardless of whether conflicts exist, for clarity. The `target_monthly_return` and `date` columns are kept as-is.

### Step 3: Merge_asof Monthly to Daily Frequency
`pd.merge_asof(direction='backward')` on `date` forward-fills monthly features onto the daily spine. Each trading day receives the most recent month-end observation on or before that date. This correctly handles weekend month-ends (e.g., if November 30 falls on a Saturday, trading days in early December still receive November's monthly data). After the merge, rows before the first valid monthly observation are dropped via `dropna(subset=monthly_cols)`. The number of trimmed rows and final date range are reported.

### Step 4: Validate
- **No duplicate dates**
- **Zero NaN in features:** all feature columns checked; any remaining NaN listed
- **Both targets checked:** daily target NaN count and monthly target NaN count
- **Daily and monthly target statistics:** mean and std for both
- **Forward-fill verification:** for a sample mid-month trading day, the most recent month-end date from the monthly table is identified and printed, confirming the merge_asof correctly matched the right monthly observation
- **Column breakdown:** daily features, monthly features (forward-filled), daily target, monthly target, date
- **Z-score sanity check:** first 3 daily features and first 3 monthly features shown with mean and std (expect mean ≈ 0, std ≈ 1)

### Step 5: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **No z-scoring applied here.** Both inputs are already z-standardised from their respective Stage 3 notebooks. This notebook is purely a merge operation.
- **`monthly_` prefix applied to all monthly features regardless of conflicts**, for clarity about feature frequency in the combined table. This is the same convention used by the combined full moments notebook.
- **`merge_asof(direction='backward')`** carries each month-end observation forward to all trading days until the next month-end arrives. The `target_monthly_return` is also carried forward -- on each trading day within month M, this target represents the return for month M+1.
- **Rows before first monthly observation dropped** via `dropna(subset=monthly_cols)`, removing trading days that precede the start of the monthly table.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_combined_means.parquet` -- keyed on `date` (trading day), containing z-scored daily features, z-scored monthly features (forward-filled via merge_asof, prefixed `monthly_`), `target_daily_return`, and `target_monthly_return`

In [2]:
# %% [markdown]
# # Stage 3 — Model-Ready: Combined Means
#
# Merges the z-scored daily means with z-scored monthly means by
# forward-filling monthly features to daily frequency.
#
# On each trading day, the model sees:
#   - Daily features: today's z-scored values (stock cwmean + macro daily)
#   - Monthly features: most recent month-end's z-scored values (stock cwmean + macro monthly)
#   - Daily target: next trading day's cap-weighted return
#   - Monthly target: next month's cap-weighted return (forward-filled)
#
# Both inputs are ALREADY z-scored — no additional standardisation needed.
#
# Input:
#   Stage_3_Model_Ready/model_market_daily_means.parquet
#   Stage_3_Model_Ready/model_market_monthly_means.parquet
#
# Output:
#   Stage_3_Model_Ready/model_market_combined_means.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')

DAILY_PATH = BASE_DIR / 'model_market_daily_means.parquet'
MONTHLY_PATH = BASE_DIR / 'model_market_monthly_means.parquet'

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD BOTH Z-SCORED TABLES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD BOTH Z-SCORED TABLES")
print("=" * 90)

daily = pd.read_parquet(DAILY_PATH)
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date').reset_index(drop=True)

print(f"\n  Daily:   {daily.shape[0]:,} rows × {daily.shape[1]} columns")
print(f"           {daily['date'].min().date()} → {daily['date'].max().date()}")

monthly = pd.read_parquet(MONTHLY_PATH)
monthly['date'] = pd.to_datetime(monthly['date'])
monthly = monthly.sort_values('date').reset_index(drop=True)

print(f"  Monthly: {monthly.shape[0]} rows × {monthly.shape[1]} columns")
print(f"           {monthly['date'].min().date()} → {monthly['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: PREFIX MONTHLY COLUMNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: PREFIX MONTHLY COLUMNS")
print("=" * 90)

# Add 'monthly_' prefix to all monthly feature columns to:
# 1. Avoid any column name conflicts with daily features
# 2. Make it clear which frequency each feature comes from

monthly_meta = ['date', 'target_monthly_return']
monthly_features = [c for c in monthly.columns if c not in monthly_meta]

# Check for actual conflicts before prefixing
daily_cols = set(daily.columns) - {'date'}
monthly_feature_set = set(monthly_features)
conflicts = daily_cols & monthly_feature_set

if conflicts:
    print(f"\n  Column conflicts found ({len(conflicts)}):")
    for c in sorted(list(conflicts))[:10]:
        print(f"    {c}")
    if len(conflicts) > 10:
        print(f"    ... and {len(conflicts) - 10} more")
else:
    print(f"\n  No column name conflicts (daily and monthly use different factor sets)")

# Prefix all monthly features regardless (clarity)
rename_map = {c: f'monthly_{c}' for c in monthly_features}
monthly = monthly.rename(columns=rename_map)

print(f"\n  Prefixed {len(rename_map)} monthly feature columns with 'monthly_'")
print(f"  Monthly columns after rename: {monthly.shape[1]}")
print(f"    Features: {len(rename_map)}")
print(f"    target_monthly_return: kept as-is")
print(f"    date: kept as-is")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: MERGE_ASOF MONTHLY TO DAILY FREQUENCY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: MERGE_ASOF MONTHLY TO DAILY FREQUENCY")
print("=" * 90)

# merge_asof(direction='backward'): for each daily date, finds the most
# recent monthly date on or before it. Handles weekend month-ends perfectly
# (e.g., Nov 30 on Saturday → Friday Dec 1 still gets November's data).

daily = daily.sort_values('date')
monthly = monthly.sort_values('date')

combined = pd.merge_asof(
    daily,
    monthly,
    on='date',
    direction='backward'
)

print(f"\n  After merge_asof: {combined.shape[0]:,} rows × {combined.shape[1]} columns")

# Identify monthly columns
monthly_cols = [c for c in combined.columns if c.startswith('monthly_')] + ['target_monthly_return']

# Drop rows before the first monthly observation
pre_trim = len(combined)
combined = combined.dropna(subset=monthly_cols).reset_index(drop=True)
trimmed = pre_trim - len(combined)

print(f"  Trimmed {trimmed} rows before first monthly observation")
print(f"  Rows: {pre_trim:,} → {len(combined):,}")
print(f"  Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: VALIDATE")
print("=" * 90)

# 4a. No duplicate dates
n_dupes = combined['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# 4b. Zero NaN in all features
all_feature_cols = [c for c in combined.columns
                    if c not in ['date', 'target_daily_return', 'target_monthly_return']]
feature_nan = combined[all_feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  ⚠ Feature NaN: {feature_nan_total}")
    for c in nan_cols.head(15).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"  ✓ Zero NaN in features")

# 4c. Target checks
daily_target_nan = combined['target_daily_return'].isna().sum()
monthly_target_nan = combined['target_monthly_return'].isna().sum()
print(f"\n  Daily target NaN:   {daily_target_nan}")
print(f"  Monthly target NaN: {monthly_target_nan}")

print(f"\n  Daily target statistics:")
print(f"    Mean:  {combined['target_daily_return'].mean():.6f}")
print(f"    Std:   {combined['target_daily_return'].std():.6f}")

print(f"\n  Monthly target statistics (forward-filled to daily):")
print(f"    Mean:  {combined['target_monthly_return'].mean():.6f}")
print(f"    Std:   {combined['target_monthly_return'].std():.6f}")

# 4d. Verify forward-fill is correct
# On any given day, the monthly features should equal the most recent month-end
sample_date = combined[combined['date'] == '2020-03-15']
if len(sample_date) == 0:
    # Pick a mid-month date that exists
    mid_month = combined[combined['date'].dt.day.between(10, 20)].iloc[0]
    sample_date_val = mid_month['date']
    sample_monthly_col = [c for c in monthly_cols if c.startswith('monthly_')][0]
    
    # Find the most recent month-end before this date
    prev_month_end = monthly[monthly['date'] <= sample_date_val]['date'].max()
    
    print(f"\n  Forward-fill verification:")
    print(f"    Sample date: {sample_date_val.date()}")
    print(f"    Most recent month-end: {prev_month_end.date() if pd.notna(prev_month_end) else 'N/A'}")

# 4e. Column breakdown
daily_feature_cols = [c for c in combined.columns 
                      if not c.startswith('monthly_') 
                      and c not in ['date', 'target_daily_return', 'target_monthly_return']]
monthly_feature_cols = [c for c in combined.columns if c.startswith('monthly_')]

print(f"\n  Column breakdown:")
print(f"    Daily features:         {len(daily_feature_cols)}")
print(f"    Monthly features:       {len(monthly_feature_cols)} (forward-filled)")
print(f"    Daily target:           1")
print(f"    Monthly target:         1 (forward-filled)")
print(f"    Date:                   1")
print(f"    Total:                  {combined.shape[1]}")

# 4f. Verify both z-scored distributions look reasonable
print(f"\n  Z-score sanity (sample daily vs monthly features):")
print(f"  {'Column':<45s} {'Mean':>8s} {'Std':>8s}")
print("  " + "-" * 65)

for c in daily_feature_cols[:3]:
    vals = combined[c].dropna()
    print(f"  {c:<45s} {vals.mean():>8.3f} {vals.std():>8.3f}")
for c in monthly_feature_cols[:3]:
    vals = combined[c].dropna()
    print(f"  {c:<45s} {vals.mean():>8.3f} {vals.std():>8.3f}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: SAVE")
print("=" * 90)

combined = combined.sort_values('date').reset_index(drop=True)

out_path = BASE_DIR / 'model_market_combined_means.parquet'
combined.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {combined.shape[0]:,} rows × {combined.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MODEL-READY COMBINED MEANS COMPLETE")
print("=" * 90)

print(f"""
  Inputs (both already z-scored):
    Daily:   model_market_daily_means.parquet   ({daily.shape[0]:,} rows × {daily.shape[1]} cols)
    Monthly: model_market_monthly_means.parquet ({monthly.shape[0]} rows × {monthly.shape[1]} cols)

  Merge strategy:
    Merge_asof join on date → forward-fill monthly columns to daily frequency
    Monthly features prefixed with 'monthly_' for clarity

  Result:
    Rows:            {combined.shape[0]:,} trading days
    Columns:         {combined.shape[1]}
      Daily features:   {len(daily_feature_cols)}
      Monthly features: {len(monthly_feature_cols)} (forward-filled)
      Targets:          2 (daily + monthly)
      Date:             1
    Dates:           {combined['date'].min().date()} → {combined['date'].max().date()}
    NaN:             {feature_nan_total} features

  Saved: {out_path}
""")

STEP 1: LOAD BOTH Z-SCORED TABLES

  Daily:   4,341 rows × 400 columns
           2007-10-02 → 2024-12-30
  Monthly: 205 rows × 326 columns
           2007-11-30 → 2024-11-30

STEP 2: PREFIX MONTHLY COLUMNS

  No column name conflicts (daily and monthly use different factor sets)

  Prefixed 324 monthly feature columns with 'monthly_'
  Monthly columns after rename: 326
    Features: 324
    target_monthly_return: kept as-is
    date: kept as-is

STEP 3: MERGE_ASOF MONTHLY TO DAILY FREQUENCY

  After merge_asof: 4,341 rows × 725 columns
  Trimmed 42 rows before first monthly observation
  Rows: 4,341 → 4,299
  Date range: 2007-11-30 → 2024-12-30

STEP 4: VALIDATE

  ✓ No duplicate dates
  ✓ Zero NaN in features

  Daily target NaN:   0
  Monthly target NaN: 0

  Daily target statistics:
    Mean:  0.000499
    Std:   0.012568

  Monthly target statistics (forward-filled to daily):
    Mean:  0.009922
    Std:   0.045127

  Forward-fill verification:
    Sample date: 2007-12-10
    Most